In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from os.path import join
import SimpleITK as sitk



import common_utils.metrics as mt
import importlib
importlib.reload(mt) 




<module 'common_utils.metrics' from '/vol/bitbucket/cc215/Projects/Cardiac_Multi_View_Segmentation/common_utils/metrics.py'>

In [2]:
def load_npy_from_med_file(med_file_path, debug=False):
    origin_image = sitk.ReadImage(med_file_path)
    original_im_arr=sitk.GetArrayFromImage(origin_image)
    original_shape=original_im_arr.shape
    pixel_spacing =origin_image.GetSpacing()
    if debug:
        print ('spacing',pixel_spacing)
        print ('shape',original_shape)
    return original_im_arr,original_shape,pixel_spacing



Load pred and gt pair

In [9]:
def evaluate(pred_dir,pred_format,gt_dir,gt_format, metrics_list =['Dice']):
    frames = ['ED','ES']
    pid_list =os.listdir(pred_dir)


    ## evaluate metric 
    n_classes = 4
    idx2cls_dict = {0:'BG',1:'LV',2:'MYO',3:'RV'}
    foreground_only=False


    metric= mt.runningMySegmentationScore(n_classes=n_classes,idx2cls_dict=idx2cls_dict,metrics_list=metrics_list)
    for frame in frames:
        for pid in sorted(pid_list):
            pred_path =join(pred_dir,pred_format.format(frame=frame,pid=pid))
            gt_path =join(gt_dir,gt_format.format(frame=frame,pid=pid))

            pred,shape,spacing = load_npy_from_med_file(pred_path)
            gt,shape,spacing = load_npy_from_med_file(gt_path)
            print ('evaluate ',str(pid),str(frame))
            metric.update(pid=pid+'_'+frame, preds=pred, gts=gt,
                                            voxel_spacing=spacing[::-1])

    print(metric.get_scores())
    df = metric.get_df()
    return df



In [4]:
UKBB_dfs={}


## Compute Metrics


In [6]:
## UKBB dataset



def evaluate_UKBB(method_name):
    pred_dir = '/vol/bitbucket/cc215/Projects/Cardiac_Multi_View_Segmentation/result/predict/{}/UKBB_test/LVSA/'.format(method_name)
    pred_format='{pid}/pred_{frame}.nii.gz'
   
    gt_dir ='/vol/medic02/users/wbai/data/cardiac_atlas/UKBB_2964/sa/test'
    gt_format='{pid}/label_sa_{frame}.nii.gz'
    df=evaluate(pred_dir,pred_format,gt_dir,gt_format)
    return df

method_name_list = [
                    'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power',
                    # 'baseline_Adam_finetune_v4_composite_50_independent_mse_adv',
                    # 'baseline_Adam_finetune_v4_composite_50_kl',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_random_random_select'
                    
                     ]
for method_name in method_name_list:
    df = evaluate_UKBB(method_name)
    UKBB_dfs[method_name] = df




evaluate  5050351 ED
evaluate  3580405 ED
evaluate  4117847 ED
evaluate  3609771 ED
evaluate  3331826 ED
evaluate  4067636 ED
evaluate  4482230 ED
evaluate  5057575 ED
evaluate  5458875 ED
evaluate  1714201 ED
evaluate  5548162 ED
evaluate  2485225 ED
evaluate  4280213 ED
evaluate  1750782 ED
evaluate  4272884 ED
evaluate  3896370 ED
evaluate  2043075 ED
evaluate  4785342 ED
evaluate  4383164 ED
evaluate  5851457 ED
evaluate  3538629 ED
evaluate  1356390 ED
evaluate  5249792 ED
evaluate  2623109 ED
evaluate  2341518 ED
evaluate  3525853 ED
evaluate  2707906 ED
evaluate  3536130 ED
evaluate  2194325 ED
evaluate  2512904 ED
evaluate  4317028 ED
evaluate  3648334 ED
evaluate  1499732 ED
evaluate  2422584 ED
evaluate  2664233 ED
evaluate  2666827 ED
evaluate  3548948 ED
evaluate  5377325 ED
evaluate  4011635 ED
evaluate  4476584 ED
evaluate  5080693 ED
evaluate  4000868 ED
evaluate  4276370 ED
evaluate  1557188 ED
evaluate  2999940 ED
evaluate  3978756 ED
evaluate  5724982 ED
evaluate  240

In [19]:
for k, df in UKBB_dfs.items():
    print(k)
    print (df.describe())

UKBB_p_value= mt.p_value_test(reference_df = UKBB_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select'],
                            test_df= UKBB_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power'])
print ('UKBB',UKBB_p_value)

baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select
       BG_Dice      LV_Dice     MYO_Dice      RV_Dice
count   1200.0  1200.000000  1200.000000  1200.000000
mean       0.0     0.935863     0.876354     0.896539
std        0.0     0.040719     0.033119     0.045701
min        0.0     0.587556     0.646893     0.669689
25%        0.0     0.923632     0.859850     0.875211
50%        0.0     0.944186     0.882942     0.905714
75%        0.0     0.964928     0.899371     0.931107
max        0.0     0.985154     0.937877     0.965549
baseline_Adam_finetune_v4_composite_50_chain_mse_random_random_select
       BG_Dice      LV_Dice     MYO_Dice      RV_Dice
count   1200.0  1200.000000  1200.000000  1200.000000
mean       0.0     0.936882     0.874823     0.897482
std        0.0     0.040229     0.034274     0.045395
min        0.0     0.575866     0.665528     0.663078
25%        0.0     0.923983     0.857367     0.874983
50%        0.0     0.945966     0.880249     0.905450

In [7]:
df.describe()



,BG_Dice,LV_Dice,MYO_Dice,RV_Dice
count,1200.0,1200.000000,1200.000000,1200.000000
mean,0.0,0.935955,0.873186,0.896453
std,0.0,0.040822,0.031172,0.045620
min,0.0,0.585522,0.690719,0.680826
25%,0.0,0.923705,0.856000,0.874371
50%,0.0,0.944210,0.877957,0.905163
75%,0.0,0.964748,0.895255,0.931143
max,0.0,0.984613,0.935025,0.966166


In [9]:
ACDC_dfs={}

In [11]:
## ACDC
ACDC_dfs={}
def evaluate_ACDC(method_name):
    pred_dir = '/vol/bitbucket/cc215/Projects/Cardiac_Multi_View_Segmentation/result/predict/{}/ACDC_all/LVSA/'.format(method_name)
    pred_format='{pid}/pred_{frame}.nrrd'
    gt_dir ='/vol/biomedic3/cc215/data/ACDC/bias_corrected_and_normalized/patient_wise'
    gt_format='{pid}/{frame}_seg.nrrd'
    df=evaluate(pred_dir,pred_format,gt_dir,gt_format)
    return df

method_name_list = [
                    'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power',
                    # 'baseline_Adam_finetune_v4_composite_50_independent_mse_adv',
                    # 'baseline_Adam_finetune_v4_composite_50_kl',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_random_random_select'

                     ]
for method_name in method_name_list:
    df = evaluate_ACDC(method_name)
    ACDC_dfs[method_name] = df

evaluate  001 ED
evaluate  002 ED
evaluate  003 ED
evaluate  004 ED
evaluate  005 ED
evaluate  006 ED
evaluate  007 ED
evaluate  008 ED
evaluate  009 ED
evaluate  010 ED
evaluate  011 ED
evaluate  012 ED
evaluate  013 ED
evaluate  014 ED
evaluate  015 ED
evaluate  016 ED
evaluate  017 ED
evaluate  018 ED
evaluate  019 ED
evaluate  020 ED
evaluate  021 ED
evaluate  022 ED
evaluate  023 ED
evaluate  024 ED
evaluate  025 ED
evaluate  026 ED
evaluate  027 ED
evaluate  028 ED
evaluate  029 ED
evaluate  030 ED
evaluate  031 ED
evaluate  032 ED
evaluate  033 ED
evaluate  034 ED
evaluate  035 ED
evaluate  036 ED
evaluate  037 ED
evaluate  038 ED
evaluate  039 ED
evaluate  040 ED
evaluate  041 ED
evaluate  042 ED
evaluate  043 ED
evaluate  044 ED
evaluate  045 ED
evaluate  046 ED
evaluate  047 ED
evaluate  048 ED
evaluate  049 ED
evaluate  050 ED
evaluate  051 ED
evaluate  052 ED
evaluate  053 ED
evaluate  054 ED
evaluate  055 ED
evaluate  056 ED
evaluate  057 ED
evaluate  058 ED
evaluate  059 

In [12]:
acdc_p_value= mt.p_value_test(reference_df = ACDC_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select'],
                            test_df= ACDC_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_random_random_select'])
print ('ACDC',acdc_p_value)

ACDC {'LV_Dice': '0.0026', 'MYO_Dice': '0.0000', 'RV_Dice': '0.1207'}


In [12]:
for k, df in ACDC_dfs.items():
    print (k)
    print(df.describe())

baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power
       BG_Dice     LV_Dice    MYO_Dice     RV_Dice
count    200.0  200.000000  200.000000  200.000000
mean       0.0    0.906004    0.808789    0.840451
std        0.0    0.076957    0.061143    0.094570
min        0.0    0.442243    0.515490    0.425032
25%        0.0    0.889149    0.782215    0.793428
50%        0.0    0.928966    0.823190    0.858517
75%        0.0    0.955271    0.852426    0.911326
max        0.0    0.973052    0.887758    0.955946


In [12]:
ACDC_dfs

{'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power':     patient_id  BG_Dice   LV_Dice  MYO_Dice   RV_Dice
 0       001_ED      0.0  0.946837  0.778550  0.877727
 1       002_ED      0.0  0.943515  0.733577  0.921918
 2       003_ED      0.0  0.965310  0.833796  0.899515
 3       004_ED      0.0  0.956802  0.783946  0.856934
 4       005_ED      0.0  0.969773  0.778315  0.857708
 ..         ...      ...       ...       ...       ...
 195     096_ES      0.0  0.897972  0.799145  0.893008
 196     097_ES      0.0  0.759555  0.745114  0.842224
 197     098_ES      0.0  0.752113  0.764306  0.783759
 198     099_ES      0.0  0.868654  0.737990  0.875208
 199     100_ES      0.0  0.844737  0.860020  0.803879
 
 [200 rows x 5 columns],
 'baseline_Adam_finetune_v4_composite_50_independent_mse_adv':     patient_id  BG_Dice   LV_Dice  MYO_Dice   RV_Dice
 0       001_ED      0.0  0.952266  0.801062  0.881033
 1       002_ED      0.0  0.960534  0.808588  0.936918
 2       003_ED      

In [26]:
## p_value

acdc_p_value= mt.p_value_test(reference_df = ACDC_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv'],test_df= ACDC_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_random'])
print ('ACDC',acdc_p_value)
ukbb_p_value = mt.p_value_test(reference_df = UKBB_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv'],test_df= UKBB_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_random'])
print('UKBB',ukbb_p_value)
MM_p_value = mt.p_value_test(reference_df = MM_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_adv'],test_df= MM_dfs['baseline_Adam_finetune_v4_composite_50_chain_mse_random'])
print('MM',MM_p_value)


ACDC {'LV_Dice': '0.0022', 'MYO_Dice': '0.0001', 'RV_Dice': '0.0596'}
UKBB {'LV_Dice': '0.1425', 'MYO_Dice': '0.0000', 'RV_Dice': '0.6409'}
MM {'LV_Dice': '0.2032', 'MYO_Dice': '0.0251', 'RV_Dice': '0.0000'}


In [ ]:
df.mean()


In [12]:
MM_dfs={}

In [14]:
## MM dataset
method_name = 'baseline_Adam_finetune_v4_composite_50_chain_mse_random' ##
MM_dfs={}
def evaluate_MM(method_name):
    gt_dir ='/vol/biomedic3/cc215/data/cardiac_MMSeg_challenge/Training-corrected/Labeled'
    gt_format='{pid}/sa_gt_{frame}.nrrd'

    pred_dir = '/vol/bitbucket/cc215/Projects/Cardiac_Multi_View_Segmentation/result/predict/{}/MM/LVSA/'.format(method_name)
    pred_format='{pid}/pred_{frame}.nrrd'
    df=evaluate(pred_dir,pred_format,gt_dir,gt_format)
    return df

method_name_list = [
                    'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_no_power',
                    # 'baseline_Adam_finetune_v4_composite_50_independent_mse_adv',
                    # 'baseline_Adam_finetune_v4_composite_50_kl',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_adv_random_select',
                    # 'baseline_Adam_finetune_v4_composite_50_chain_mse_random_random_select'

                     ]
for method_name in method_name_list:
    df = evaluate_MM(method_name)
    MM_dfs[method_name] = df



df.mean()

evaluate  A0S9V9 ED
evaluate  A1D0Q7 ED
evaluate  A1D9Z7 ED
evaluate  A1E9Q1 ED
evaluate  A1O8Z3 ED
evaluate  A2C0I1 ED
evaluate  A2N8V0 ED
evaluate  A3B7E5 ED
evaluate  A3H1O5 ED
evaluate  A4B5U4 ED
evaluate  A4J4S4 ED
evaluate  A4U9V5 ED
evaluate  A5E0T8 ED
evaluate  A6B5G9 ED
evaluate  A6D5F9 ED
evaluate  A6M1Q7 ED
evaluate  A7D9L8 ED
evaluate  A7G0P5 ED
evaluate  A7M7P8 ED
evaluate  A7O4T6 ED
evaluate  A8C9U8 ED
evaluate  A8E1F4 ED
evaluate  A9C5P4 ED
evaluate  A9E3G9 ED
evaluate  A9J5Q7 ED
evaluate  A9J8W7 ED
evaluate  B0I2Z0 ED
evaluate  B0N3W8 ED
evaluate  B2C2Z7 ED
evaluate  B2D9M2 ED
evaluate  B2D9O2 ED
evaluate  B2F4K5 ED
evaluate  B2G5R2 ED
evaluate  B3D0N1 ED
evaluate  B3O1S0 ED
evaluate  B3P3R1 ED
evaluate  B4O3V3 ED
evaluate  B6D0U7 ED
evaluate  B8H5H6 ED
evaluate  B8J7R4 ED
evaluate  B9E0Q1 ED
evaluate  B9O1Q0 ED
evaluate  C0K1P0 ED
evaluate  C0S7W0 ED
evaluate  C1G5Q0 ED
evaluate  C1K8P5 ED
evaluate  C2J0K3 ED
evaluate  C2L5P7 ED
evaluate  C2M6P8 ED
evaluate  C3I2K3 ED


NameError: name 'MM_dfs' is not defined

In [15]:
MM_dfs[method_name]=df
df.describe()

,BG_Dice,LV_Dice,MYO_Dice,RV_Dice
count,300.0,300.000000,300.000000,300.000000
mean,0.0,0.875117,0.766908,0.824853
std,0.0,0.119429,0.113244,0.119989
min,0.0,0.000000,0.000000,0.183657
25%,0.0,0.857906,0.735551,0.799986
50%,0.0,0.906018,0.790092,0.858719
75%,0.0,0.939673,0.833039,0.896651
max,0.0,0.981484,0.892920,0.967891


,BG_Dice,LV_Dice,MYO_Dice,RV_Dice
count,300.0,300.000000,300.000000,300.000000
mean,0.0,0.893491,0.802149,0.835294
std,0.0,0.074856,0.068768,0.102410
min,0.0,0.154555,0.240272,0.062953
25%,0.0,0.874611,0.773181,0.798983
50%,0.0,0.908166,0.813498,0.863184
75%,0.0,0.938024,0.846573,0.898019
max,0.0,0.982844,0.895662,0.966797


In [ ]:
df.describe()
